In [ ]:
!pip install selenium

In [ ]:
from selenium import webdriver  # 引入webdriver，webdriver為瀏覽器主控權
from selenium.webdriver.chrome.service import Service  # 用來管理driver，例如安裝路徑
from selenium.webdriver.chrome.options import Options  # 用來作Chrome設定
from selenium.webdriver.common.by import By  # 用來指定元素的定位方式
from selenium.webdriver.support.ui import WebDriverWait  # 等待後才執行
from selenium.webdriver.support import expected_conditions as EC  # 描述持續等待的條件
import time
import random  # 延遲操作
import calendar
from datetime import datetime

**肉眼觀察出網址規律後，以函式計算並回傳指定月區間的url串列。**

In [ ]:
def get_uri(region, year, month):
    """由於spotify chart網站設定“當周週五-下週四”為一週的榜單計算範疇，故定義一函式，
    用以確定目前這一個月有幾個星期五，而有幾個星期五就代表當月有幾週榜單、
    應該要下載幾次csv檔，同時也能知道每週的榜單結束哪一個週四，根據先前調查，
    週四的日期會連動到url網址串。所以我們可以推算url，並包裝在list中回傳。"""
    output = []
    cal_in_lst_of_lsts = calendar.monthcalendar(year, month)
    today_day = datetime.now().day
    for a_week in cal_in_lst_of_lsts:
        if a_week[3] != 0 and a_week[3] <= today_day: # [3]代表週日、[3] <= today_day是用於避免拜訪對還未公開榜單的周次之網站
            url = f'https://charts.spotify.com/charts/view/regional-{region}-weekly/{year}-{month:02}-{a_week[3]:02}'
            output.append(url)
    return output


**制定模擬人類點擊csv下載鈕的函式**

In [ ]:
def download_btn_click():
    # 模擬捲動頁面
    driver.execute_script("window.scrollTo(0, 500);")
    time.sleep(random.uniform(2, 4))

    # 點擊CSV下載
    csv_btn = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, '#__next > div > div:nth-child(3) > div > div > div.styled__CSVLink-sc-135veyd-5.hJopSD > span > span > button')))
    csv_btn.click()
    print("CSV下載觸發")
    time.sleep(random.uniform(4, 8))  # 考量下載速度可能因當下網速而不一，所以設定睡覺秒數較久
    return '完成該地區下載。'

**Chromedriver設定**

In [ ]:
# 設定webdriver的位置。讓selenium知道它接手chrome後，要去哪裡找到chromedriver，並且使用chromedriver來執行後續操作。
services = Service(
    executable_path="自己本地端的資料夾/chromedriver")

# 新建空設定
options = Options()

# 設定下載檔案的存放路徑，並加到options物件中
download_dir = '自己本地端的資料夾'
prefs = {
    'download.default_directory': download_dir,
    'savefile.default_directory': download_dir,  # 兩個都要設定
    'download.prompt_for_download': False,  # 禁止詢問下載提示
    'download.directory_upgrade': True,
    'safebrowsing.enabled': True  # 安全瀏覽設定
}
options.add_experimental_option('prefs', prefs)

**啟動瀏覽器，並等待人類主動完成登入後，接手網頁操作**
爬蟲步驟


*   爬取global 2025/10~2025/12月數據
*   再爬取Japan 2025/10~2025/12月數據
*   再爬取South Korea 2025/10~2025/12月數據 若有錯誤則顯示停在哪個頁面上



In [ ]:
# options&service備妥，可建立瀏覽器控制物件，並啟動瀏覽器。
driver = webdriver.Chrome(options=options, service=services)
driver.maximize_window()

driver.get('https://charts.spotify.com/home')

input('已在spotify chart網站入口，請手動登入後輸入enter: ') # 手動登入
wait = WebDriverWait(driver, 10)

# 接手操作當前頁面元素
try:
    regions = ['jp', 'kr', 'global']
    year = 2025
    start_month = 10
    end_month = 12
    for reg in regions:
        for month in range(start_month, end_month+1):
            urls = get_uri(reg, year=year, month=month)
            for url in urls:
              print(url)
              driver.get(url)
              print('已進入網頁, 尋找下載按鈕')
              download_btn_click()
except Exception as e:
    print(f'Error {e}，停留在當前頁面{driver.current_url}...')
finally:
    print('finished')
    driver.close()